### Run inference Anthropic

Required inputs:

file_name = "interactions_output_file.parquet" <- parquet file containing all interactions

directory = "images" <- directory containing interaction images

api = "..." <- your own Anthropic api 

In [ ]:
file_name = "interactions_output_file.parquet" 
directory = "images"
api = "YOUR_ANTHROPIC_API_KEY"

Libraries:

In [ ]:
import base64
import time
from pathlib import Path
import math
import pandas as pd
import anthropic
import os

In [ ]:
client = anthropic.Anthropic(
    api_key=api
)

MODELS = [
    "claude-sonnet-4-6",  # latest Claude 4.6 ...
]

def normalize_image_path(image_path) -> Path:
    if image_path is None:
        raise ValueError("image_path is None")

    if isinstance(image_path, float) and math.isnan(image_path):
        raise ValueError("image_path is NaN")

    image_path = str(image_path).strip()
    if not image_path:
        raise ValueError("image_path is empty")

    return Path(image_path).expanduser().resolve()


def image_to_base64(image_path: str) -> tuple[str, str]:
    path = normalize_image_path(image_path)

    # Faster and slightly more explicit than full stat-based logic
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Image not found or not a file: {path}")

    suffix = path.suffix.lower()
    mime_map = {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".webp": "image/webp",
    }
    mime_type = mime_map.get(suffix)
    if mime_type is None:
        raise ValueError(f"Unsupported image type: {suffix}")

    with open(path, "rb") as f:
        image_b64 = base64.b64encode(f.read()).decode("utf-8")

    return image_b64, mime_type


def ask_claude(model_name: str, image_path: str, caption: str) -> str:
    image_b64, mime_type = image_to_base64(image_path)

    response = client.messages.create(
        model=model_name,
        temperature=0.0,
        max_tokens=50,
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": mime_type,
                            "data": image_b64,
                        },
                    },
                    {
                        "type": "text",
                        "text": caption,
                    },
                ],
            }
        ]
    )

    return response.content[0].text.strip()


df = pd.read_parquet(file_name,     
                    engine="pyarrow",
                    dtype_backend="pyarrow")

required_cols = ["fileName", "caption"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

results = []

for idx, row in df.iterrows():
    image_path = row["fileName"]
    caption = row["caption"]
    ground_truth = row["ground_truth"] if "ground_truth" in df.columns else None

    print(f"\nRow {idx}")
    print("Image:", image_path)
    print("Caption:", caption)

    for model_name in MODELS:
        try:
            prediction = ask_claude(
                model_name=model_name,
                image_path=image_path,
                caption=caption,
            )

            result_row = {
                "index": idx,
                "model": model_name,
                "fileName": row["fileName"],
                "question": caption,
                "prediction": prediction,
                "status": "ok",
            }


            if ground_truth is not None:
                result_row["ground_truth"] = ground_truth

            results.append(result_row)
            print(prediction)
            print(f"[{model_name}] OK")

        except Exception as e:
            result_row = {
                "index": idx,
                "model": model_name,
                "fileName": row["fileName"],
                "caption": caption,
                "prediction": None,
                "status": f"error: {e}",
            }

            if ground_truth is not None:
                result_row["ground_truth"] = ground_truth

            results.append(result_row)
      
            print(f"[{model_name}] ERROR: {e}")

        time.sleep(0.5)

results_df = pd.DataFrame(results)
results_df.to_csv("anthropic_inference_results.csv", index=False)

print("\nDone.")
print(results_df.head())